# Signals dataset builder - ML training dataset example

This notebook walks through the Signals **dataset builder**: turning your Snowplow event data into a **labeled, point-in-time-correct** training dataset for a machine learning model.

**Scenario:** predict whether an ecommerce session will convert (a `transaction`) from behaviour seen *so far* in the session. Features are computed from the same attribute-group definitions that power real-time Signals, so training and serving use identical feature logic.

**Why point-in-time correctness matters:** at serving time the model only sees events that have happened so far in the session. The builder computes each feature using only events *before* the anchor timestamp, which prevents data leakage (the most common cause of models that test well but fail in production).

**Workflow:**
1. Define attribute groups with the features to learn from
2. Submit a dataset run (session anchors or custom anchors)
3. Poll for completion and retrieve results
4. Optionally inspect / save the generated SQL

Docs: https://docs.snowplow.io/docs/signals/ml-training-datasets/

## 1. Install the SDK

In [ ]:
%pip install snowplow-signals==0.4.7 pandas

## 2. Connect to Signals

Loads the four Signals credentials from [Google Colab secrets](https://x.com/GoogleColab/status/1719798406195867814) and creates the `Signals` client. The client submits dataset runs and retrieves results via the Signals API.

Add the following secrets in the Colab sidebar (key icon): `SP_API_URL`, `SP_API_KEY`, `SP_API_KEY_ID`, `SP_ORG_ID`.

In [ ]:
from google.colab import userdata
from snowplow_signals import Signals

sp_signals = Signals(
    api_url=userdata.get("SP_API_URL"),
    api_key=userdata.get("SP_API_KEY"),
    api_key_id=userdata.get("SP_API_KEY_ID"),
    org_id=userdata.get("SP_ORG_ID"),
)
sp_signals

## 3. Define the features

Each attribute becomes a column in the training dataset. Here we count product views and add-to-cart actions within the session, filtered from the ecommerce action event by its `type` property.

You don't need to publish these to build a dataset - the builder works directly from the definitions. Publishing is what you'd do later to serve the same features in real time.

In [ ]:
from datetime import timedelta
from snowplow_signals import Attribute, Event, EventProperty, Criteria, Criterion

# The Snowplow ecommerce action event carries an action `type` (product_view, add_to_cart, transaction, ...)
ecommerce_action = Event(
    name="snowplow_ecommerce_action",
    vendor="com.snowplowanalytics.snowplow.ecommerce",
    version="1-0-2",
)


def action_type_is(value: str) -> Criteria:
    """Filter ecommerce action events to a single action type."""
    return Criteria(
        all=[
            Criterion.eq(
                EventProperty(
                    vendor="com.snowplowanalytics.snowplow.ecommerce",
                    name="snowplow_ecommerce_action",
                    major_version=1,
                    path="type",
                ),
                value,
            )
        ]
    )


product_view_count = Attribute(
    name="product_view_count",
    description="Product views in the session so far",
    events=[ecommerce_action],
    aggregation="counter",
    type="int32",
    period=timedelta(hours=1),
    criteria=action_type_is("product_view"),
)

add_to_cart_count = Attribute(
    name="add_to_cart_count",
    description="Add-to-cart actions in the session so far",
    events=[ecommerce_action],
    aggregation="counter",
    type="int32",
    period=timedelta(hours=1),
    criteria=action_type_is("add_to_cart"),
)

### Group the features

Attributes live inside an attribute group, keyed by `domain_sessionid` so features are computed per session.

In [ ]:
from snowplow_signals import StreamAttributeGroup, domain_sessionid

ecommerce_group = StreamAttributeGroup(
    name="ecommerce_session_features",
    version=1,
    attribute_key=domain_sessionid,
    owner="jack.keene@snowplowanalytics.com",
    attributes=[product_view_count, add_to_cart_count],
)

## 4. Submit a dataset run with session anchors

`submit_dataset_run_with_session_anchors()` submits a dataset build for server-side execution. It scans every session in the training window and labels it: sessions where the **goal** (a `transaction`) occurred get a positive anchor (label=1) at the goal event; sessions without it get negative anchors (label=0) at random events, then downsampled to `max_negative_ratio`.

Key knobs used below:
- `min_events=3` - skip anchors with too little prior signal
- `max_anchors_per_session=10` - up to 10 training rows per session, so long sessions don't dominate
- `max_negative_ratio=1.0` - a balanced 1:1 positive/negative dataset

This returns a `DatasetRunResponse` immediately. The dataset is built asynchronously on the server.

In [ ]:
from datetime import datetime, timezone
from snowplow_signals import TrainingSpan

run = sp_signals.submit_dataset_run_with_session_anchors(
    attribute_groups=[ecommerce_group],
    goal_criteria=action_type_is("transaction"),
    training_span=TrainingSpan(
        start_time=datetime(2024, 1, 1, tzinfo=timezone.utc),
        end_time=datetime(2024, 4, 1, tzinfo=timezone.utc),
    ),
    min_events=3,
    max_anchors_per_session=10,
    max_negative_ratio=1.0,
)
run

## 5. Poll for completion

The dataset is built server-side. Poll `get_dataset_run_status()` until the status is `success` or `failed`.

In [ ]:
import time
from snowplow_signals import DatasetRunStatus

while True:
    status = sp_signals.get_dataset_run_status(run.id)
    print(f"Status: {status.status}")
    if status.status == DatasetRunStatus.SUCCESS:
        break
    if status.status == DatasetRunStatus.FAILED:
        raise RuntimeError("Dataset run failed")
    time.sleep(5)

## 6. Get the training dataset as a DataFrame

`get_dataset_run_preview()` fetches a preview of the completed dataset. Call `to_pandas()` on the result to get a DataFrame (one row per anchor; columns for the attribute key, anchor timestamp, label, and every attribute). Set `limit` up to 10,000 rows.

In [ ]:
preview = sp_signals.get_dataset_run_preview(run.id, limit=10000)
df = preview.to_pandas()
print(df.shape)
df.head()

## Alternative: user-supplied anchors

If you already have a labeled anchor table (columns: your attribute key e.g. `domain_sessionid`, `anchor_ts`, and optionally `label`), use `submit_dataset_run_with_custom_anchors()` instead of the session-anchor method. Everything downstream (polling, preview, `to_pandas`) is identical.

```python
from snowplow_signals import WarehouseTable

run = sp_signals.submit_dataset_run_with_custom_anchors(
    attribute_groups=[ecommerce_group],
    anchors_table=WarehouseTable(
        database="analytics",
        schema="ml",
        table="my_anchor_events",
    ),
    anchors_have_label=True,
)
```

## Inspect the generated SQL

If you want to review the SQL before execution, use `build_dataset_with_session_anchors()` to generate a `DatasetBundle` and save it to disk:

```python
bundle = sp_signals.build_dataset_with_session_anchors(
    attribute_groups=[ecommerce_group],
    goal_criteria=action_type_is("transaction"),
    training_span=TrainingSpan(
        start_time=datetime(2024, 1, 1, tzinfo=timezone.utc),
        end_time=datetime(2024, 4, 1, tzinfo=timezone.utc),
    ),
)
bundle.save_to("./dataset_output")
```